In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TCT(nn.Module):
    def __init__(self, text_dim=768, audio_dim=74, visual_dim=128, num_heads=2):
        super().__init__()
        self.text_dim = text_dim

        # Projection layers for each modality
        self.audio_proj = nn.Linear(audio_dim, text_dim)
        self.visual_proj = nn.Linear(visual_dim, text_dim)

        # Multi-head attention components
        self.query_proj = nn.Linear(text_dim, text_dim)
        self.key_proj = nn.Linear(text_dim, text_dim)
        self.value_proj = nn.Linear(text_dim, text_dim)

        # Normalization and feed-forward
        self.layer_norm1 = nn.LayerNorm(text_dim)
        self.layer_norm2 = nn.LayerNorm(text_dim)
        self.ffn = nn.Sequential(
            nn.Linear(text_dim, 4 * text_dim),
            nn.ReLU(),
            nn.Linear(4 * text_dim, text_dim)
        )

        # Output projection
        self.output_proj = nn.Linear(text_dim, text_dim)

    def forward(self, text, audio, visual):
        # Project audio and visual features
        audio_proj = self.audio_proj(audio).unsqueeze(1)
        visual_proj = self.visual_proj(visual).unsqueeze(1)

        # Compute tensor products (Cartesian space)
        T_a = torch.matmul(text, audio_proj.transpose(1, 2))
        T_v = torch.matmul(text, visual_proj.transpose(1, 2))
        T_av = (T_a + T_v) / 2  # Average fusion

        # Prepare queries, keys, values
        Q = self.query_proj(text)
        K_av = self.key_proj(T_av )
        V_av = self.value_proj(T_av )

        # Scaled dot-product attention
        attn_weights = F.softmax(torch.matmul(Q, K_av.transpose(1, 2)) / (self.text_dim ** 0.5), dim=-1)
        attn_output = torch.matmul(attn_weights, V_av)

        # Residual connection and layer norm
        output = self.layer_norm1(text + attn_output)

        # Feed-forward network
        ffn_output = self.ffn(output)
        output = self.layer_norm2(output + ffn_output)

        # Final projection
        output = self.output_proj(output)

        return output